In [35]:
#Getting the data
# The `car_fuel_efficiency_2026.csv` file is already in the working directory (`homework`).
import pandas as pd
import numpy as np

car_data = pd.read_csv("car_fuel_efficiency_2026.csv")

car_data


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


## Q1: finding the column with missing values

In [36]:
# Q1 loading the selected columns

selected_columns = ['engine_displacement',
                    'horsepower',
                    'vehicle_weight',
                    'model_year',
                    'fuel_efficiency_mpg']

selected_car_data = car_data[selected_columns]

selected_car_data.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [37]:
#find the missing values
missing_values = selected_car_data.isnull().sum()
missing_values

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

As can be seen above the `horsepower` column contains 877 missing values.

## Q2: finding the median horsepower

In [38]:
# Q2 finding the median of the horsepower data
median_hp = selected_car_data['horsepower'].median()
median_hp

np.float64(254.0)

The median horsepower is 254 hp.

In [39]:
# Work only with the five columns specified by the homework.
df = selected_car_data.copy()
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [40]:
#Lets create the training data using zero and mean  imputation
train_mean = df_train['horsepower'].mean()
df_train_zero = df_train.fillna(0)
df_val_zero = df_val.fillna(0)
df_train_mean = df_train.fillna(train_mean)
df_val_mean = df_val.fillna(train_mean)

The following function can be used to estimate the Root Mean Square Error (RMSE)

## Q3 Preparation

In [41]:
#a generic function to estimate the Root Mean Square Error (RMSE)

def rmse(y,y_pred):
    return (np.sqrt(np.mean((y - y_pred) ** 2))
)

In [42]:
y_train = df_train_zero["fuel_efficiency_mpg"].values
y_val = df_val_zero["fuel_efficiency_mpg"].values

X_train = df_train_zero.drop(columns="fuel_efficiency_mpg").values
X_val = df_val_zero.drop(columns="fuel_efficiency_mpg").values

## Q3: fill missing values with zero vs mean

In [43]:
# Function to train the linear regression model
def train_linear_regression(X, y, r=0.0):
    X = np.column_stack([np.ones(len(X)), X])
    XTX = X.T @ X + r * np.eye(X.shape[1])
    w = np.linalg.solve(XTX, X.T @ y)
    return w[0], w[1:]



# function to train the model and get the RMSE score
def evaluate(df_train, df_val, fill_value, r=0.0):
    X_train = df_train.drop(columns='fuel_efficiency_mpg').fillna(fill_value).values
    y_train = df_train['fuel_efficiency_mpg'].values
    X_val = df_val.drop(columns='fuel_efficiency_mpg').fillna(fill_value).values
    y_val = df_val['fuel_efficiency_mpg'].values
    w0, w = train_linear_regression(X_train, y_train, r=r)
    return rmse(y_val, w0 + X_val @ w)

In [45]:
# compare filling the missing horsepower with zero and the train mean
score_zero = evaluate(df_train, df_val, fill_value=0)
score_mean = evaluate(df_train, df_val, fill_value=train_mean)

print('zero:', round(score_zero, 3))
print('mean:', round(score_mean, 3))

zero: 2.205
mean: 2.202


## Q4: regularization

In [46]:
# Try different regularization values
regularization_values = [0, 0.01, 0.1, 1, 5, 10, 100]
q4_scores = {}
for r in regularization_values:
    score = evaluate(df_train, df_val, fill_value=0, r=r)
    q4_scores[r] = score
    print(r, round(score, 4))

print('best r:', min(q4_scores, key=q4_scores.get))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292
best r: 0


## Q5: checking the model stability with different seeds

In [47]:
# calculating validation RMSE using seeds 0 to 9
scores = []
for seed in range(10):
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    train = df.iloc[idx[:n_train]]
    val = df.iloc[idx[n_train:n_train + n_val]]
    scores.append(evaluate(train, val, fill_value=0))

print(scores)
print('std:', round(np.std(scores), 3))

[np.float64(2.2392041430507406), np.float64(2.2021128210838423), np.float64(2.163419778753154), np.float64(2.2043588768529694), np.float64(2.2018800943320533), np.float64(2.245785150908184), np.float64(2.269721207951665), np.float64(2.1907945999354204), np.float64(2.2166112016944384), np.float64(2.2070365928234077)]
std: 0.029


## Q6: final test score

In [48]:
# split using seed 9 and combine train + validation data
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]
df_train_full = pd.concat([df_train, df_val])

X_train_full = df_train_full.drop(columns='fuel_efficiency_mpg').fillna(0).values
y_train_full = df_train_full['fuel_efficiency_mpg'].values
X_test = df_test.drop(columns='fuel_efficiency_mpg').fillna(0).values
y_test = df_test['fuel_efficiency_mpg'].values

w0, w = train_linear_regression(X_train_full, y_train_full, r=0.001)
print('test rmse:', round(rmse(y_test, w0 + X_test @ w), 3))

test rmse: 2.236
